In [1]:
import pandas as pd
from datasets import load_dataset
pd.set_option('display.max_colwidth', None)

In [2]:
DETESTS_DIS_REPO_URL = "CLiC-UB/DETESTS-Dis"
LEVEL_4_DETESTS_DIS = "CLiC-UB/DETESTS-Dis-level4"

detests_dataset = load_dataset(DETESTS_DIS_REPO_URL, split="train")
level_4_detests_dataset = load_dataset(LEVEL_4_DETESTS_DIS, split="train")

detests_dataset_df = detests_dataset.to_pandas()
level_4_detests_df = level_4_detests_dataset.to_pandas()

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def iterar_df_interactivo(df):
    """
    Muestra una interfaz interactiva para navegar fila por fila por un DataFrame.
    """
    # Reseteamos el índice por si el dataframe está filtrado
    df_reset = df.reset_index(drop=True)
    
    index = 0
    max_idx = len(df_reset) - 1
    
    # Crear los widgets (botones, texto y área de salida)
    out = widgets.Output()
    btn_prev = widgets.Button(description="Anterior", icon="arrow-left")
    btn_next = widgets.Button(description="Siguiente", icon="arrow-right")
    lbl_info = widgets.Label()
    
    def update_view():
        with out:
            clear_output(wait=True) # Limpia el output anterior
            lbl_info.value = f" Fila {index} de {max_idx} "
            # Mostramos la fila actual como un DataFrame de 1 fila para mantener el formato de tabla
            display(df_reset.iloc[[index]])
            
    def on_prev(b):
        nonlocal index
        if index > 0:
            index -= 1
            update_view()
            
    def on_next(b):
        nonlocal index
        if index < max_idx:
            index += 1
            update_view()
            
    # Asignar los eventos de click a los botones
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    
    # Agrupar los controles y mostrarlos
    controles = widgets.HBox([btn_prev, lbl_info, btn_next])
    display(widgets.VBox([controles, out]))
    
    # Inicializar la primera vista
    update_view()

In [4]:
iterar_df_interactivo(level_4_detests_df)

In [20]:
iterar_df_interactivo(detests_dataset_df.loc[(detests_dataset_df["source"] == "detests") & (detests_dataset_df["stereotype"] == 1) & (detests_dataset_df["implicit"] == 0),["id", "text", "stereotype", "level1", "level2", "level3", "level4", "stereotype_soft"]])

In [25]:
detests_dataset_df.loc[detests_dataset_df["comment_id"] == "d_213"]

,source,id,comment_id,text,level1,level2,level3,level4,stereotype_a1,stereotype_a2,stereotype_a3,stereotype,stereotype_soft,implicit_a1,implicit_a2,implicit_a3,implicit,implicit_soft
644,detests,d_213_01,d_213,aqui le damos una paguita al pobre negro de los cojones,0,0,0,20190716_CR,1,1,1,1,0.9526,0,1,0,0,0.2689


In [27]:
level_4_detests_df.loc[level_4_detests_df["id"] == "20190716_CR"]

,id,text
29,20190716_CR,Un 'cortacabezas' en la patera.


In [32]:
detests_filtered = detests_dataset_df[detests_dataset_df["source"] == "detests"]
comments_sizes = detests_filtered.groupby("comment_id").size()

print("Distribution of rows per comment_id:")
display(comments_sizes.describe())

Distribution of rows per comment_id:


count    2548.000000
mean        2.209184
std         1.839614
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        24.000000
dtype: float64

In [40]:
def get_stereohoax_thread_average(df):
    """
    Calculates the average length of the threads on 'stereohoax' source.
    Since level3 refers to the first comment_id of the thread, grouping by level3
    gives us the size (number of comments) of each thread.
    """
    stereohoax_data = df[df["source"] == "stereohoax"]
    
    # Agrupamos por la columna 'level3' que identifica el hilo
    thread_sizes = stereohoax_data.groupby("level3").size()
    
    return thread_sizes.mean(), thread_sizes

avg_len, thread_sizes = get_stereohoax_thread_average(detests_dataset_df)
print(f"Average thread length on stereohoax: {avg_len:.2f}")

display(thread_sizes.describe())

Average thread length on stereohoax: 6.54


count    654.000000
mean       6.539755
std       43.147783
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max      750.000000
dtype: float64